<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

# Linked Entity with Knowledge Graph

This notebook demonstrates how to:
1. Extract named entities from a text using spaCy
2. Link entities to Wikidata IDs using the Wikidata API
3. Discover relationships between entities using SPARQL queries on Wikidata

## Overview

The workflow processes an input text to:
- **Extract**: Named entities (people, places, organizations) using spaCy
- **Link**: Map entities to Wikidata Q-codes via the Wikipedia/Wikidata search API
- **Reason**: Discover semantic relationships between entities through Wikidata's knowledge graph using SPARQL

## Dependencies

- `requests`: HTTP library for API calls
- `spacy`: NLP library for entity extraction
- `itertools`: For generating entity pair permutations

In [ ]:
# Import required libraries
import requests
import spacy
from itertools import permutations

## WikiData Knowledge Graph Class

This class provides methods to interact with Wikidata's APIs for entity lookup and relationship discovery.

- **Wikidata Search API (`wbsearchentities`)**: Translates natural language entity names to Wikidata Q-codes
- **Wikidata Query Service (`sparql`)**: Executes SPARQL queries to find relationships between entities based on their Wikidata IDs

In [ ]:
class WikiDataKG:
    """
    Wrapper for Wikidata APIs to link entities and discover relationships.
    """
    
    def __init__(self) -> 'WikiDataKG':
        """
        Initialize the WikiDataKG instance with a User-Agent header.
        The User-Agent is required by Wikidata's API to identify the client.
        """
        self.header = {"User-Agent": "entity-demo/0.1 (you@example.com)"}
        
    # 1. Link each entity to a Wikidata ID (top search hit)
    def link(self, name: str) -> Optional[tuple[str, str]]:
        """
        Search Wikidata for a named entity and return its ID and description.
        
        Args:
            name (str): Natural language name of the entity (e.g., 'Uruguay')
            
        Returns:
            tuple[str, str] | None: (Wikidata Q-id, description) if found, None otherwise
        """
        # Call Wikidata's Wikipedia/Wikidata search API
        r = requests.get("https://www.wikidata.org/w/api.php", headers=self.header, params={
            "action": "wbsearchentities",  # Action: search entities
            "search": name,                # Query string to search for
            "language": "en",             # Language preference
            "format": "json",             # Return JSON format
            "limit": 1})                   # Only return the top result
        # check http request status
        if not r.ok:
            return None
            
        hits = r.json()["search"]
        
        # Extract ID and optional description from first result
        return (hits[0]["id"], hits[0].get("description", "")) if hits else None

    # 2. Ask Wikidata the relation between two entities
    def relations(self, a: str, b: str) -> List[str]:
        """
        Query Wikidata for direct relationships from item a to item b.

        Args:
            a (str): Wikidata Q-id of the source item.
            b (str): Wikidata Q-id of the target item.

        Returns:
            List[str]: Relationship labels, or an empty list when none are found
                or the service response cannot be read.
        """
        # Reject malformed IDs before interpolating them into a SPARQL query.
        if not (a.startswith("Q") and a[1:].isdigit() and b.startswith("Q") and b[1:].isdigit()):
            raise ValueError("Both entity IDs must be Wikidata Q-ids, such as 'Q1335'.")

        query = f"""
        SELECT ?propLabel
        WHERE {{
          wd:{a} ?wdt wd:{b} .
          ?prop wikibase:directClaim ?wdt .
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}"""

        try:
            response = requests.get(
                "https://query.wikidata.org/sparql",
                headers={**self.header, "Accept": "application/sparql-results+json"},
                params={"query": query, "format": "json"},
                timeout=30,
            )
            # Raise an HTTP error for rate limits, server errors, and other bad statuses.
            response.raise_for_status()
            payload = response.json()
            return [
                binding["propLabel"]["value"]
                for binding in payload["results"]["bindings"]
                if "propLabel" in binding and "value" in binding["propLabel"]
            ]
        except requests.exceptions.RequestException as exc:
            print(f"Wikidata request failed for {a} -> {b}: {exc}")
        except (ValueError, KeyError, TypeError) as exc:
            # ValueError also covers a non-JSON response body (e.g., a proxy error page).
            preview = getattr(locals().get("response"), "text", "")[:300]
            print(f"Could not parse Wikidata response for {a} -> {b}: {exc}; response begins: {preview!r}")

        return []


## Main Workflow

This section demonstrates the complete pipeline:
1. Load spaCy's large English model for Named Entity Recognition (NER)
2. Process example text and extract entities
3. Link extracted entities to Wikidata
4. Find relationships between all pairs of linked entities

In [ ]:
# Initialize the WikiData knowledge graph and spaCy NER model
kg = WikiDataKG()
nlp = spacy.load('en_core_web_lg')  # Large English model for better entity accuracy

In [ ]:
# Input text describing Montevideo, Uruguay, and historical events
text = """Montevideo was founded by the Spanish in the early 18th century as a military stronghold. Uruguay won its independence in 1828 following a three-way struggle between Spain, Argentina and Brazil. It is a constitutional democracy, where the president fulfills the roles of both head of state and head of government"""

# Step 1: Extract named entities from text using spaCy's NER
doc = nlp(text)

print("Entities:")
for ent in doc.ents:
    print(f"entity: '{ent.text}' has label: {ent.label_}")

### Mapping Entities to Wikidata

Only entities with specific labels (GPE, LOC, ORG, PERSON) are mapped to Wikidata. This filters out less informative entities like dates and cardinal numbers.

In [ ]:
# Step 2: Link each entity to Wikidata (skip if already processed)
linked = {}
for ent in doc.ents:
    # Only map entities of type GPE (GeoPolitical Entity), LOC (Location), 
    # ORG (Organization), or PERSON; avoid duplicates
    if ent.label_ in ("GPE", "LOC", "ORG", "PERSON") and ent.text not in linked:
        #print(ent.text)
        result = kg.link(ent.text)
        if result:
            linked[ent.text] = result
            print(f"{ent.text:15} -> {result[0]:8} {result[1]}")

In [ ]:
linked

### Discovering Entity Relationships

For each pair of entities, we query Wikidata to discover the semantic relationships between them, building a knowledge graph.

In [ ]:
# Step 3: For all pairs of linked entities, find their relationships
# Uses permutations to generate unique entity pairs (a, b)
for (na, (qa, _)), (nb, (qb, _)) in permutations(linked.items(), 2):
    rels = kg.relations(qa, qb)
    if rels:
        print(f"{na} --{', '.join(rels)}--> {nb}")

# Student Exercises: NER, Entity Linking, and Knowledge Graphs

Use the spaCy pipeline (`nlp`) and Wikidata helper (`kg`) defined above. Run the notebook from top to bottom first. Keep in mind that NER predictions and Wikidata search results can be wrong; inspect outputs critically.

## Exercise 1 — Compare spaCy and GLiNER entity extraction

Run both extractors on the same passage. Compare the entity spans and categories. Which model finds entities the other misses? Do their label systems match exactly?


In [ ]:
# Reuse the Montevideo passage and spaCy pipeline defined earlier.
# Import GLiNER, a model for extracting entities and relations from text.
from gliner import GLiNER

# Load a pretrained relation-extraction model (the first run may download it).
model = GLiNER.from_pretrained("knowledgator/gliner-relex-large-v1.0")

comparison_text = text
spacy_entities = [(ent.text, ent.label_) for ent in nlp(comparison_text).ents]
print("spaCy:")
for span, label in spacy_entities:
    print(f"  {span} ({label})")

# GLiNER uses natural-language label candidates rather than spaCy label codes.
gliner_labels = ["person", "organization", "location", "date", "nationality"]
gliner_entities = model.predict_entities(comparison_text, gliner_labels, threshold=0.5)
print("\nGLiNER:")
for entity in gliner_entities:
    print(f"  {entity['text']} ({entity['label']})")


## Exercise 2 — Add a new passage and inspect NER labels

Write a short passage containing at least five named entities from different categories (for example people, organizations, and places). Use spaCy to extract them. Identify one false positive or missed entity, then explain why it may have occurred.


In [ ]:
# TODO: Replace this sample with your own passage containing several entities.
student_text = "Ada Lovelace worked with Charles Babbage in London on early computing ideas."
student_doc = nlp(student_text)

# Print the recognized text spans and spaCy entity labels.
for ent in student_doc.ents:
    print(f"{ent.text:20} {ent.label_:10} {spacy.explain(ent.label_)}")


## Exercise 3 — Evaluate NER with a hand-labeled reference

Manually create a gold set of entity spans and labels for your passage. Compare it with spaCy predictions using exact text-and-label matches. Calculate precision, recall, and F1. Explain how differences in span boundaries or label choice affect these metrics.


In [ ]:
# TODO: Update these expected annotations to match your own passage and judgment.
gold = {("Ada Lovelace", "PERSON"), ("Charles Babbage", "PERSON"), ("London", "GPE")}
predicted = {(ent.text, ent.label_) for ent in student_doc.ents}

# Compute exact-match evaluation counts and metrics.
tp = len(gold & predicted)
fp = len(predicted - gold)
fn = len(gold - predicted)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print(f"TP={tp}, FP={fp}, FN={fn}")
print(f"Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")


## Exercise 4 — Link entities and build a small graph

Take the unique location or person entities from your new passage and try linking them to Wikidata. Review each returned description to decide whether the candidate is correct. Then query relationships between linked items. What makes an automatic entity linker ambiguous?

### Optional challenge

Modify the pipeline to compare Wikidata links for spaCy and GLiNER entity spans. De-duplicate repeated mentions and report which spans could not be linked. Add a short note about API limitations, response errors, or rate limits.


In [ ]:
# Try linking unique person and location mentions from your student passage.
candidate_labels = {"PERSON", "GPE", "LOC"}
student_linked = {}
for ent in student_doc.ents:
    if ent.label_ in candidate_labels and ent.text not in student_linked:
        result = kg.link(ent.text)
        if result:
            student_linked[ent.text] = result
            print(f"{ent.text:20} -> {result[0]} | {result[1]}")
        else:
            print(f"{ent.text:20} -> no Wikidata match found")

# Query direct Wikidata relations for each unordered pair, in both directions.
linked_items = list(student_linked.items())
for i in range(len(linked_items)):
    for j in range(i + 1, len(linked_items)):
        name_a, (qid_a, _) = linked_items[i]
        name_b, (qid_b, _) = linked_items[j]
        for source_name, source_qid, target_name, target_qid in [
            (name_a, qid_a, name_b, qid_b),
            (name_b, qid_b, name_a, qid_a),
        ]:
            relation_labels = kg.relations(source_qid, target_qid)
            if relation_labels:
                print(f"{source_name} --{', '.join(relation_labels)}--> {target_name}")
